# Lower RL Training Analysis

Analysis notebook for the lower gNB-1 agent trained with behavior cloning followed by TD3 fine-tuning.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
DATASET = ROOT / 'datasets/lower_bc_dataset_controlled_gnb1.npz'
BC_LOG = ROOT / 'logs/lower_bc_pretrain_log.csv'
RL_LOG = ROOT / 'logs/lower_rl_training_log.csv'
EVAL_CSV = ROOT / 'logs/lower_rl_eval.csv'

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

def require(path):
    if not path.exists():
        print(f'Missing {path}. Run the corresponding training/evaluation command first.')
        return False
    return True

print(ROOT)

/home/oussama/Desktop/chech


## Dataset Stats

In [2]:
if require(DATASET):
    data = np.load(DATASET, allow_pickle=True)
    obs = data['observations']
    actions = data['actions']
    scenarios = pd.Series(data['scenario_names'].astype(str), name='scenario')
    sources = pd.Series(data['demand_load_source'].astype(str), name='demand_source')
    print({'samples': len(obs), 'obs_dim': obs.shape[1], 'action_dim': actions.shape[1]})
    display(scenarios.value_counts().rename_axis('scenario').reset_index(name='samples'))
    display(sources.value_counts().rename_axis('demand_source').reset_index(name='samples'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(obs.reshape(-1), bins=80)
    axes[0].set_title('Observation values')
    axes[1].hist(actions.reshape(-1), bins=np.arange(-6.5, 7.5, 1.0))
    axes[1].set_title('Expert action offsets (dB)')
    axes[1].set_xlabel('offset dB')
    plt.show()

Missing /home/oussama/Desktop/chech/datasets/lower_bc_dataset_controlled_gnb1.npz. Run the corresponding training/evaluation command first.


## BC Pretraining Loss

In [3]:
if require(BC_LOG):
    bc = pd.read_csv(BC_LOG)
    display(bc.tail())
    ax = bc.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o')
    ax.set_ylabel('MSE on normalized offsets')
    ax.set_title('Behavior cloning loss')
    plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_bc_pretrain_log.csv. Run the corresponding training/evaluation command first.


## RL Reward Curve

In [4]:
if require(RL_LOG):
    rl = pd.read_csv(RL_LOG)
    display(rl.tail())
    if 'reward' in rl:
        rl['reward_smooth'] = rl['reward'].rolling(200, min_periods=1).mean()
        ax = rl.plot(x='timestep', y=['reward', 'reward_smooth'], alpha=0.8)
        ax.set_title('TD3 lower-agent reward')
        ax.set_ylabel('step reward')
        plt.show()
    ep = rl.dropna(subset=['episode_return']) if 'episode_return' in rl else pd.DataFrame()
    if not ep.empty:
        ax = ep.plot(x='timestep', y='episode_return', marker='o')
        ax.set_title('Episode return during TD3 fine-tuning')
        plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_rl_training_log.csv. Run the corresponding training/evaluation command first.


## Evaluation Summary

In [5]:
if require(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV)
    display(ev.head())
    summary_cols = ['episode_return', 'handover_count', 'pingpong_count', 'sla_severity', 'final_total_demand_imbalance_std', 'action_vs_expert_mae_mean']
    summary_cols = [c for c in summary_cols if c in ev]
    summary = ev.groupby('mode')[summary_cols].agg(['mean', 'std']).round(4)
    display(summary)
    ax = ev.groupby('mode')['episode_return'].mean().plot(kind='bar')
    ax.set_title('Mean evaluation return')
    ax.set_ylabel('return')
    plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_rl_eval.csv. Run the corresponding training/evaluation command first.


## Demanded PRB Total Load Per gNB

In [6]:
if require(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV)
    total_cols = [c for c in ev.columns if c.startswith('demand_total_g')]
    if total_cols:
        totals = ev.groupby('mode')[total_cols].mean().T
        display(totals.round(4))
        ax = totals.plot(kind='bar')
        ax.set_title('Final total demanded PRB load per gNB')
        ax.set_xlabel('gNB')
        ax.set_ylabel('demanded PRB load')
        plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_rl_eval.csv. Run the corresponding training/evaluation command first.


## Per-Slice Demanded PRB Load

In [7]:
if require(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV)
    slice_cols = [c for c in ev.columns if c.startswith('demand_g') and not c.startswith('demand_total')]
    if slice_cols:
        for mode, grp in ev.groupby('mode'):
            means = grp[slice_cols].mean()
            records = []
            for col, value in means.items():
                _, gnb, slice_type = col.split('_', 2)
                records.append({'gNB': gnb, 'slice': slice_type, 'load': value})
            pivot = pd.DataFrame(records).pivot(index='gNB', columns='slice', values='load').fillna(0.0)
            ax = pivot.plot(kind='bar', stacked=True)
            ax.set_title(f'Final per-slice demanded PRB load: {mode}')
            ax.set_ylabel('demanded PRB load')
            plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_rl_eval.csv. Run the corresponding training/evaluation command first.


## Handovers, Ping-Pong, SLA Severity

In [8]:
if require(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV)
    cols = [c for c in ['handover_count', 'pingpong_count', 'sla_severity'] if c in ev]
    if cols:
        means = ev.groupby('mode')[cols].mean()
        display(means.round(4))
        axes = means.plot(kind='bar', subplots=True, layout=(1, len(cols)), figsize=(5 * len(cols), 4), legend=False)
        plt.tight_layout()
        plt.show()

Missing /home/oussama/Desktop/chech/logs/lower_rl_eval.csv. Run the corresponding training/evaluation command first.


## Action Distribution

In [9]:
if require(DATASET):
    data = np.load(DATASET, allow_pickle=True)
    actions = data['actions']
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.boxplot([actions[:, i] for i in range(actions.shape[1])], labels=[f'a{i}' for i in range(actions.shape[1])])
    ax.set_title('Expert action offset distribution by controlled action dimension')
    ax.set_ylabel('offset dB')
    plt.show()

if require(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV)
    if 'action_vs_expert_mae_mean' in ev:
        ax = ev.groupby('mode')['action_vs_expert_mae_mean'].mean().plot(kind='bar')
        ax.set_title('Mean action deviation from expert')
        ax.set_ylabel('MAE dB')
        plt.show()

Missing /home/oussama/Desktop/chech/datasets/lower_bc_dataset_controlled_gnb1.npz. Run the corresponding training/evaluation command first.
Missing /home/oussama/Desktop/chech/logs/lower_rl_eval.csv. Run the corresponding training/evaluation command first.
